In [1]:
import threading

import requests
import pandas as pd
from bs4 import BeautifulSoup
import io

In [2]:
# Getting all the indexes for the players
import string
player_indexes = [f"{s}-index" for s in string.ascii_lowercase]

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/114.0.0.0 Safari/537.36",
}
session = requests.Session()
session.headers.update(headers)

In [7]:
# The player pages

import time
import random
import re


player_weblinks = set()
for index in player_indexes:
    url = f"https://www.sports-reference.com/cbb/players/{index}.html"
    # Wait between 5 to 15 seconds randomly
    time.sleep(random.uniform(5, 15))
    
    response = session.get(url, timeout= 10)
    if response.status_code == 429:
        print("Rate limit exceeded")
        break
    soup = BeautifulSoup(response.text, "html.parser")

    # Each player link is in <p><a href="/cbb/players/{slug}.html">
    for div in soup.find_all("div"):
        paragraphs = div.find_all("p")
        for p in paragraphs:
            if p:
                player_link = p.find("a")
                note = p.find("small", class_="note")
            else:
                continue
            
            if note:
                year_string = note.get_text(strip=True)
                year_string = year_string.strip()  # Remove leading/trailing spaces
                
                # Split on dash and take the first part
                match = re.search(r"(\d{4})", year_string)
                if match and int(match.group(1)) < 1999:
                    continue
                school_link = note.find("a")
                if school_link and "women" in school_link["href"]:
                    continue
            else:
                continue
            if player_link:
                href = player_link["href"]
                if href.startswith("/cbb/players/") and href.endswith(".html"):
                    player_weblinks.add(href)

player_weblinks

['/cbb/players/menghe-anyam-1.html',
 '/cbb/players/jordan-aaberg-1.html',
 '/cbb/players/karl-aaker-1.html',
 '/cbb/players/mike-aaman-2.html',
 '/cbb/players/carlton-aaron-1.html',
 '/cbb/players/jordan-aaron-1.html',
 '/cbb/players/nate-aaron-1.html',
 '/cbb/players/shaddean-aaron-1.html',
 '/cbb/players/shaqquan-aaron-1.html',
 '/cbb/players/troy-aaron-1.html',
 '/cbb/players/joe-aase-1.html',
 '/cbb/players/brian-abadie-1.html',
 '/cbb/players/ronell-abaekobe-1.html',
 '/cbb/players/eli-abaev-1.html',
 '/cbb/players/kwesi-abakah-1.html',
 '/cbb/players/kevin-abanobi-1.html',
 '/cbb/players/shota-abashidze-1.html',
 '/cbb/players/churchill-abass-1.html',
 '/cbb/players/malik-abbott-1.html',
 '/cbb/players/matthew-abbott-1.html',
 '/cbb/players/tara-abbott-1.html',
 '/cbb/players/ty-abbott-1.html',
 '/cbb/players/huthifah-abdeljawad-1.html',
 '/cbb/players/mostafa-abdel-latif-1.html',
 '/cbb/players/malek-abdelgowad-1.html',
 '/cbb/players/fowzi-abdelsamad-1.html',
 '/cbb/players/sa

In [8]:
len(player_weblinks)

141633

In [17]:
from pathlib import Path

dirpath = Path("../data/raw_data/College_Stats/")
dirpath.mkdir(parents=True, exist_ok=True)
filepath = Path(dirpath / "player_links.txt")
if not filepath.exists():
    with filepath.open("w") as f:
        for link in player_weblinks:
            f.write(f"{link}\n")

In [7]:
import re
def slug_to_name(slug):
    name = re.sub(r"-\d+$", "", slug)
    return name.replace("-", " ").title()

In [26]:
def helper(table, drafted, weight, height):
    try:
        df = pd.read_html(io.StringIO(str(table)))[0]
        df = pl.from_pandas(df)
    except Exception:
        return pl.DataFrame({})

    # Get name
    slug = player.split("/cbb/players/")[1].replace(".html", "")
    
    # Commenting out so we can identify nulls in Pos for aggregate vs by-season data
    # if "Pos" in df.columns:
    #     df = df.with_columns(
    #         pl.col("Pos").fill_null(strategy="forward")
    #     )
    df = df.with_columns(
        pl.lit(slug).alias("Name"),
        pl.lit("Drafted" if drafted else "Not drafted").alias("Drafted"),
        pl.lit(weight).alias("Weight"),
        pl.lit(height).alias("Height")
    )
    return df

In [27]:
len(player_weblinks)


46947

In [11]:
og_player_weblinks = player_weblinks.copy()

In [12]:
player_weblinks: set = set(player_weblinks)

In [30]:
per_df_list = []
advanced_df_list = []
total_df_list = []

processed_weblinks = set()

In [31]:
import time 
import sys

def countdown_timer(seconds):
    for remaining in range(seconds, 0, -1):
        mins, secs = divmod(remaining, 60)
        timer_display = f"\r Sleeping ... {mins:02d}:{secs:02d} remaining"
        sys.stdout.write(timer_display)
        sys.stdout.flush()
        time.sleep(1)
    print("\r Done Sleeping! Trying Again")

In [ ]:
import polars as pl
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import threading
from datetime import datetime

# Config
DELAY_EVERY = 1000
MAX_WORKERS = 4
DELAY_PER_THREAD = 1

# Shared State
processed_count = 0
count_lock = threading.Lock()
rate_limit_event = threading.Event()
countdown_lock = threading.Lock()

temp_weblinks = player_weblinks - processed_weblinks

def scrape_players(player, index):
    global processed_count
    
    stagger = (index %  MAX_WORKERS) * DELAY_PER_THREAD
    print(f"[{player}] Initial stagger: {stagger:.1f}s")
    time.sleep(stagger)
    
    url = f"https://www.sports-reference.com/{player}"
    while True:

        print(f"Processing {player}...")

        try:
            # Wait between 5 to 30 seconds randomly
            time.sleep(random.uniform(5, 30))
            
            response = session.get(url, timeout=10)
            if response.status_code == 429:
                
                if count_lock.acquire(blocking=False):
                    print(f"\n[{player}] Got 429! Triggering global rate limit.")
                    try:
                        rate_limit_event.clear()
                        print("\n[GLOBAL] Sleeping for 30 minutes due to rate limit.")
                        countdown_timer(30 * 60)  # Block for 15 mins
                        print("\n[GLOBAL] Finished sleeping. Unblocking threads.")
                        rate_limit_event.set()
                    finally:
                        count_lock.release()
                else:
                    time.sleep(random.uniform(0.5, 2.5))  # prevent reburst
                    if not rate_limit_event.is_set():
                        rate_limit_event.wait()
                        time.sleep(random.uniform(0.5, 2.5))  # prevent reburst
                continue
            elif response.status_code != 200:
                print(f"\nFailed with status {response.status_code}, skipping.")
            else:
                soup = BeautifulSoup(response.text, "html.parser")
                # Locate table with stats (per-season stats table has id="players_per_game")
                meta_div = soup.find("div", id="meta")
    
                # Check if any link in the meta div contains 'draft' in the href
                drafted = any("draft" in a.get("href", "") for a in meta_div.find_all("a"))
                weight = None
                height = None
                for p in meta_div.find_all("p"):
                    span = [span.text for span in p.find_all("span")]
                    if span:
                        height = span[0]
                        weight = span[1] if len(span) > 1 else None
                bottom_nav = soup.find("div", id="bottom_nav_container")
                nba_link_text = None
                if bottom_nav:
                    for link in bottom_nav.find_all("a"):
                        if link.get_text(strip=True) == "Basketball-Reference.com":
                            nba_link_text = link.get_text(strip=True)
                            break
                if nba_link_text is None:
                    raise ValueError("Did not play in NBA")
                
                # Get Per Game
                table = soup.find("table", id="players_per_game")
                df = helper(table, drafted, weight, height)
                if not df.is_empty():
                    per_df_list.append(df)
                
                # Totals
                table = soup.find("table", id="players_totals")
                df = helper(table, drafted, weight, height)
                if not df.is_empty():
                    total_df_list.append(df)
                
                # Advanced
                table = soup.find("table", id="players_advanced")
                df = helper(table, drafted, weight, height)
                if not df.is_empty():
                    advanced_df_list.append(df)
                
                print(f"\nSuccessfully parsed: {player}")
        except ValueError as e:
            print(f"\nError: {e}. Skipping: {player}")
            player_weblinks.remove(player)
            break
        except Exception as e:
            print(f"\nError: {e}. Try again record: {player}")
            continue        
        break
    
    with count_lock:
        processed_count += 1
        processed_weblinks.add(player)
        if processed_count % DELAY_EVERY == 0:
            extra_sleep = random.uniform(60, 180)
            print(f"[{processed_count}] Taking a long break: {extra_sleep:.1f}s")
            time.sleep(extra_sleep)

if temp_weblinks:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(scrape_players, player, i) for i, player in enumerate(iter(temp_weblinks))]
        for _ in tqdm(as_completed(futures), total=len(futures)):
            pass
else:
    print("You need to run the higher cell as there are no more weblinks available or you are simply finished scraping :)")

[/cbb/players/dikembe-dixson-1.html] Initial stagger: 0.0s
Processing /cbb/players/dikembe-dixson-1.html...
[/cbb/players/arsh-mattu-1.html] Initial stagger: 1.0s
[/cbb/players/mohamed-bendary-1.html] Initial stagger: 2.0s
[/cbb/players/codey-mcelroy-1.html] Initial stagger: 3.0s


  0%|          | 0/46871 [00:00<?, ?it/s]

Processing /cbb/players/arsh-mattu-1.html...
Processing /cbb/players/mohamed-bendary-1.html...
Processing /cbb/players/codey-mcelroy-1.html...

Error: HTTPSConnectionPool(host='www.sports-reference.com', port=443): Read timed out. (read timeout=10). Try again record: /cbb/players/arsh-mattu-1.html
Processing /cbb/players/arsh-mattu-1.html...

Error: HTTPSConnectionPool(host='www.sports-reference.com', port=443): Read timed out. (read timeout=10). Try again record: /cbb/players/dikembe-dixson-1.html
Processing /cbb/players/dikembe-dixson-1.html...

Error: Did not play in NBA. Skipping: /cbb/players/mohamed-bendary-1.html
[/cbb/players/eric-kindler-1.html] Initial stagger: 0.0s
Processing /cbb/players/eric-kindler-1.html...


  0%|          | 2/46871 [00:31<183:45:08, 14.11s/it]


Error: Did not play in NBA. Skipping: /cbb/players/codey-mcelroy-1.html
[/cbb/players/drew-timme-1.html] Initial stagger: 1.0s

Error: Did not play in NBA. Skipping: /cbb/players/arsh-mattu-1.html
[/cbb/players/kenya-carruthers-1.html] Initial stagger: 2.0s
Processing /cbb/players/drew-timme-1.html...
Processing /cbb/players/kenya-carruthers-1.html...


  0%|          | 4/46871 [00:38<94:52:39,  7.29s/it] 

... (5,625 outputs left out)

  6%|▌         | 2917/46871 [4:32:08<32:50:50,  2.69s/it]


Error: Did not play in NBA. Skipping: /cbb/players/cody-cambre-1.html
[/cbb/players/xavier-brown-1.html] Initial stagger: 0.0s
Processing /cbb/players/xavier-brown-1.html...
Processing /cbb/players/sam-fernley-1.html...
Processing /cbb/players/nancy-marrero-1.html...


  6%|▌         | 2918/46871 [4:32:18<61:58:33,  5.08s/it]


Error: Did not play in NBA. Skipping: /cbb/players/xavier-brown-1.html
[/cbb/players/david-kyles-1.html] Initial stagger: 1.0s
Processing /cbb/players/david-kyles-1.html...


In [87]:

total_df_list

ERROR! Session/line number was not unique in database. History logging moved to new session 95


[shape: (5, 35)
 ┌─────────┬─────────┬─────────┬─────────┬───┬──────────────┬─────────────┬────────┬────────┐
 │ Season  ┆ Team    ┆ Conf    ┆ Class   ┆ … ┆ Name         ┆ Drafted     ┆ Weight ┆ Height │
 │ ---     ┆ ---     ┆ ---     ┆ ---     ┆   ┆ ---          ┆ ---         ┆ ---    ┆ ---    │
 │ str     ┆ str     ┆ str     ┆ str     ┆   ┆ str          ┆ str         ┆ str    ┆ str    │
 ╞═════════╪═════════╪═════════╪═════════╪═══╪══════════════╪═════════════╪════════╪════════╡
 │ 2017-18 ┆ Clemson ┆ ACC     ┆ FR      ┆ … ┆ aaron-bond-1 ┆ Not drafted ┆ 245lb  ┆ 6-8    │
 │ 2018-19 ┆ Clemson ┆ ACC     ┆ SO      ┆ … ┆ aaron-bond-1 ┆ Not drafted ┆ 245lb  ┆ 6-8    │
 │ 2019-20 ┆ Clemson ┆ ACC     ┆ JR      ┆ … ┆ aaron-bond-1 ┆ Not drafted ┆ 245lb  ┆ 6-8    │
 │ 2020-21 ┆ Clemson ┆ ACC     ┆ SR      ┆ … ┆ aaron-bond-1 ┆ Not drafted ┆ 245lb  ┆ 6-8    │
 │ 2017-18 ┆ 2017-18 ┆ 2017-18 ┆ 2017-18 ┆ … ┆ aaron-bond-1 ┆ Not drafted ┆ 245lb  ┆ 6-8    │
 └─────────┴─────────┴─────────┴─────────┴──

IOStream.flush timed out


In [11]:
advanced_df_list

[]

In [10]:
per_df_list

[shape: (2, 35)
 ┌─────────┬────────────┬─────────┬─────────┬───┬───────────────────┬─────────────┬────────┬────────┐
 │ Season  ┆ Team       ┆ Conf    ┆ Class   ┆ … ┆ Name              ┆ Drafted     ┆ Weight ┆ Height │
 │ ---     ┆ ---        ┆ ---     ┆ ---     ┆   ┆ ---               ┆ ---         ┆ ---    ┆ ---    │
 │ str     ┆ str        ┆ str     ┆ str     ┆   ┆ str               ┆ str         ┆ str    ┆ str    │
 ╞═════════╪════════════╪═════════╪═════════╪═══╪═══════════════════╪═════════════╪════════╪════════╡
 │ 2007-08 ┆ Washington ┆ Pac-10  ┆ FR      ┆ … ┆ thomas-abercrombi ┆ Not drafted ┆ 191lb  ┆ 6-6    │
 │         ┆ State      ┆         ┆         ┆   ┆ e-1               ┆             ┆        ┆        │
 │ 2007-08 ┆ 2007-08    ┆ 2007-08 ┆ 2007-08 ┆ … ┆ thomas-abercrombi ┆ Not drafted ┆ 191lb  ┆ 6-6    │
 │         ┆            ┆         ┆         ┆   ┆ e-1               ┆             ┆        ┆        │
 └─────────┴────────────┴─────────┴─────────┴───┴─────────────────

In [ ]:
def build_df(list_df):
    master_df = pl.concat(list_df, how="diagonal_relaxed")
    master_df = master_df.with_columns(
        pl.col("Name").map_elements(slug_to_name, return_dtype=pl.String)
    )
    return master_df

In [ ]:
dirpath = Path("../data/raw_data/College_Stats")
dirpath.mkdir(parents=True, exist_ok=True)

df = build_df(per_df_list)
filepath = Path(dirpath / "master_list_per_game.csv")
df.write_csv(filepath, index=False)

df = build_df(total_df_list)
filepath = Path(dirpath / "master_list_total.csv")
df.write_csv(filepath, index=False)

df = build_df(advanced_df_list)
filepath = Path(dirpath / "master_list_advanced.csv")
df.write_csv(filepath, index=False)

In [119]:
master_df = master_df.drop(["Unnamed: 14"])

In [128]:
master_df.filter((pl.col("Pos").is_null()) & (pl.col("Season") == '2000-01')).unique()

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
"""2000-01""",null,36.0,2.0,22.9,3.5,8.0,0.438,0.3,1.8,3.2,6.3,0.511,0.457,2.4,4.3,0.568,6.0,1.5,1.2,1.0,2.4,2.4,9.8,"""Gerald Wallace""","""Drafted""",0.175,2.3,3.7


In [130]:
master_df.filter(pl.col("Name") == "Gerald Wallace")

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
"""2000-01""",null,36.0,2.0,22.9,3.5,8.0,0.438,0.3,1.8,3.2,6.3,0.511,0.457,2.4,4.3,0.568,6.0,1.5,1.2,1.0,2.4,2.4,9.8,"""Gerald Wallace""","""Drafted""",0.175,2.3,3.7


In [134]:
# Fixing Gerald Wallace
master_df = master_df.with_columns(
    pl.when(pl.col("Name") == "Gerald Wallace")
    .then(pl.lit("SF"))
    .otherwise(pl.col("Pos"))
    .alias("Pos")
)
master_df.filter(pl.col("Name") == "Gerald Wallace")

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64
"""2000-01""","""SF""",36.0,2.0,22.9,3.5,8.0,0.438,0.3,1.8,3.2,6.3,0.511,0.457,2.4,4.3,0.568,6.0,1.5,1.2,1.0,2.4,2.4,9.8,"""Gerald Wallace""","""Drafted""",0.175,2.3,3.7


In [135]:
# Dropping Null Pos
master_df = master_df.drop_nulls(["Pos"])

In [136]:
master_df.null_count()

Season,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,2P,2PA,2P%,eFG%,FT,FTA,FT%,TRB,AST,STL,BLK,TOV,PF,PTS,Name,Drafted,3P%,ORB,DRB
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,5,929,688,9,42,41,1243,1244,1243,1244,1243,1242,9,17,16,103,478,792,791,837,54,4,0,0,1387,1935,1935


In [138]:
master_df = master_df.with_columns(
    pl.col("Season").str.slice(0, 4).cast(pl.Int32).alias("Year")
)

In [145]:
# Filtering for above 1999
master_df = master_df.filter(pl.col("Year") >= 2000).sort(pl.col("Year"), descending=True)

In [147]:
master_df.select(pl.col("Year").value_counts())

Year
struct[2]
"{2021,1}"
"{2018,1}"
"{2017,2}"
"{2015,16}"
"{2014,21}"
…
"{2004,47}"
"{2003,56}"
"{2002,51}"


In [148]:

dirpath = Path("../data/raw_data/College_Stats")
dirpath.mkdir(parents=True, exist_ok=True)
filepath = Path(dirpath / "master_list_updated_above_2000.csv")
if not filepath.exists():
    master_df.write_csv(filepath, separator=",")